# Optimisation Methods

This notebook defines the methods used in our experiments. Every method needs **NumPy** (`import numpy as np`). The other method-specific libraries are:

- **BO (`"BO"`)** uses `run_bo`. Install `scikit-optimize`; the function imports `skopt.gp_minimize` and `skopt.space.Real`.
- **CMA-ES (`"CMA-ES"`)** uses `run_cma_es`. Install `cma`; the function imports `cma`.
- **TuRBO-1 (`"TuRBO-1"`)** uses `run_turbo1`. Install `torch`, `gpytorch`, and Uber Research's TuRBO. The function imports `torch` and `turbo.Turbo1`.
- **Random (`"Random"`)** uses `run_uniform_random`.


## Calling the method

Each method has the following arguments:

```python
result = run_bo(objective, bounds, budget, seed, params)
```

Change the function name to `run_cma_es`, `run_turbo1`, or `run_uniform_random` to call another method. You can also use `run_method`, explained at the end of this notebook.

- **`objective`** is the Python function to minimise. It receives one point and returns one finite number. 
- **`bounds`** gives one `[lower, upper]` pair for each coordinate. Every lower bound must be smaller than its upper bound. Lists and NumPy arrays both work. For example, `[[-4.0, 4.0], [-4.0, 4.0]]` lets both coordinates range from -4 to 4.
- **`budget`** is a positive integer giving the total number of objective calls, including initial samples and restarts. For example, `budget=50` allows 50 calls per method.
- **`seed`** is a small nonnegative integer controlling the method's random choices. 
- **`params`** is a dictionary containing settings for the chosen method. Each method's required keys and an example appear below. Random uses an empty dictionary, `{}`.

The helper functions are listed as below:

- `_validated_bounds(bounds)` converts bounds to a NumPy array and checks their shape, finite values, and ordering. It returns the checked array or raises `ValueError`.
- `EvaluationResults(objective, budget, bounds)` creates an initially empty evaluation record. Calling this object as `recorder(point)` checks the point, calls the objective once, records the point and value, and returns that value.
- `recorder.result(seconds, diagnostics=None)` returns the result dictionary described above, possible to call `diagnostics` for extra information.
- `BudgetReached` is an exception used to stop CMA-ES or TuRBO before an extra objective call exceeds the budget. 


In [ ]:
import time
import numpy as np

# The recorder raises this to stop an optimiser before it exceeds the shared budget.
class BudgetReached(Exception):
    pass

def _validated_bounds(bounds):
    """Return finite [lower, upper] pairs as an array."""
    bounds = np.asarray(bounds, dtype=float)
    
    # defensive checks on bounds 
    if bounds.ndim != 2 or bounds.shape[1] != 2 or len(bounds) == 0:
        raise ValueError("bounds must have shape (dimension, 2)")
    if not np.isfinite(bounds).all() or np.any(bounds[:, 0] >= bounds[:, 1]):
        raise ValueError("Bounds must be finite, with lower < upper")
    
    return bounds


class EvaluationResults:
    """Record evaluation results."""
    def __init__(self, objective, budget, bounds):
        # Require a positive integer budget before storing the validated bounds.
        if not isinstance(budget, (int, np.integer)) or budget <= 0:
            raise ValueError("budget must be a positive integer")
        
        self.objective = objective
        self.budget = int(budget)
        self.bounds = _validated_bounds(bounds)
        # Keep points and objective values in evaluation order for the later trajectory plots.
        self.X, self.y = [], []

    # call actual function and record results
    def __call__(self, x):
        # Check before calling the objective so rejected extra calls are never evaluated.
        if len(self.y) >= self.budget:
            raise BudgetReached
        
        point = np.asarray(x, dtype=float)
        
        # defensive check
        if point.shape != (len(self.bounds),) or not np.isfinite(point).all():
            raise ValueError("Invalid objective point")
        if np.any(point < self.bounds[:, 0]) or np.any(point > self.bounds[:, 1]):
            raise ValueError("Point is outside the bounds")
        
        # evaluate the objective
        # Pass a copy so an objective cannot modify the point stored in the history.
        value = np.asarray(self.objective(point.copy()))
        if value.ndim != 0 or not np.isfinite(value):
            raise ValueError("Objective must return a finite scalar")
    
        self.X.append(point.copy())
        self.y.append(float(value))
        return float(value)
    
    def result(self, seconds, diagnostics = None):
        # All methods must return the same number of evaluations for a fair comparison.
        if len(self.y) != self.budget:
            raise RuntimeError(f"Incomplete run: {len(self.y)} of {self.budget} evaluations")
        
        # Copy any supplied diagnostics so adding the evaluation count does not alter the caller's dictionary.
        if diagnostics is None:
             diagnostics = {}
        else:
            diagnostics = diagnostics.copy()
        diagnostics["evaluations"] = len(self.y)
        
        return {
            "X": np.asarray(self.X), 
            "y": np.asarray(self.y),
            "seconds": float(seconds),
            "diagnostics": diagnostics
        }


## Bayesian optimisation: `run_bo`

BO builds a model of the objective from the values it has seen, then uses that model to decide where to evaluate next. This function calls scikit-optimize's `gp_minimize`. It needs NumPy and **scikit-optimize**. Its `params` dictionary must contain these five keys:

- **`n_initial_points`** is a positive integer. It sets how many points are evaluated before model-guided search and must not exceed `budget`.
- **`initial_point_generator`** is a string, here `"random"`. It chooses how the initial points are generated.
- **`acq_func`** is a string, here `"EI"`, selecting the rule for the next point. EI means expected improvement.
- **`noise`** is a nonnegative number, here `1e-10`, giving the assumed observation-noise variance. This tiny value is used for the deterministic objective.
- **`xi`** is a nonnegative number, here `0.0`, giving EI's improvement offset. Zero lets EI consider small improvements near the precision targets.


```python
bo_params = {
    "n_initial_points": 4,
    "initial_point_generator": "random",
    "acq_func": "EI",
    "noise": 1e-10,
    "xi": 0.0,
}
result = run_bo(objective, bounds, budget=50, seed=42, params=bo_params)
```

This uses `objective` and `bounds` from the shared example and returns the same result dictionary. Reference: [gp_minimize documentation](https://scikit-optimize.readthedocs.io/en/stable/modules/generated/skopt.gp_minimize.html).


In [ ]:
def run_bo(objective, bounds, budget, seed, params):
    """Run Gaussian-process Bayesian optimisation and return the evaluation history."""
    from skopt import gp_minimize
    from skopt.space import Real

    bounds = _validated_bounds(bounds)
    recorder = EvaluationResults(objective, budget, bounds)
    started = time.perf_counter()
    # The recorder acts as the objective, saving every initial and model-guided evaluation.
    gp_minimize(
        func=recorder,
        # Create one continuous search interval per coordinate.
        dimensions=[Real(float(lo), float(hi)) for lo, hi in bounds],
        # The total includes the initial random points, rather than adding them to the budget.
        n_calls=budget,
        n_initial_points=params["n_initial_points"],
        initial_point_generator=params["initial_point_generator"],
        acq_func=params["acq_func"],
        noise=params["noise"],
        xi=params["xi"],
        random_state=seed,
    )
    return recorder.result(time.perf_counter() - started)


## CMA-ES

`run_cma_es` calls `cma.fmin2` in unit coordinates. It draws the initial
mean uniformly and records evaluations in the original domain.

- **`sigma0`** (required) sets the positive initial step size in unit coordinates. A value of 0.25 corresponds to 2 units per coordinate on `[-4, 4]`.
- **`seed_offset`** (required) is an integer added to the seed for the first start. Each subsequent start increments the internal seed by one.
- **`popsize`** (optional) sets the number of candidates per full generation. It must be an integer of at least 2. Omit it or use `None` for PyCMA's default.

```python
cma_params = {"sigma0": 0.25, "seed_offset": 1, "popsize": 8}
result = run_cma_es(objective, bounds, budget=50, seed=42, params=cma_params)
```

If PyCMA terminates early, the wrapper draws a new mean and uses the
remaining calls. All starts share the same budget. If the budget ends
partway through a generation, the evaluations already made count toward
the budget and can supply the best solution. Diagnostics record the number
of restarts and the evaluation at which each start began.

In [ ]:
def run_cma_es(objective, bounds, budget, seed, params):
    """Run CMA-ES with an optional population size and a fixed evaluation budget."""
    bounds = _validated_bounds(bounds)
    recorder = EvaluationResults(objective, budget, bounds)
    lower, upper = bounds[:, 0], bounds[:, 1]
    # Use the library default unless a valid population size is supplied.
    popsize = params.get("popsize")
    if popsize is not None and (
        isinstance(popsize, (bool, np.bool_))
        or not isinstance(popsize, (int, np.integer))
        or popsize < 2
    ):
        raise ValueError("popsize must be an integer of at least 2 or None")

    import cma

    # CMA-ES searches [0, 1] in each coordinate; map proposals back to the original bounds.
    def normalized_objective(point):
        return recorder(lower + (upper - lower) * np.asarray(point))

    started = time.perf_counter()
    rng = np.random.default_rng(seed)
    starts_at = []
    # If a start converges early, restart with a new mean and spend only the remaining calls.
    while len(recorder.y) < budget:
        before = len(recorder.y)
        options = {
            "bounds": [0.0, 1.0],
            "maxfevals": int(budget - before),
            # Give each start a reproducible, distinct seed.
            "seed": seed + params["seed_offset"] + len(starts_at),
            # Skip the extra final-mean evaluation so it does not consume an unplanned call.
            "eval_final_mean": False,
            "verbose": -9,
            # Disable CMA-ES log files; the experiment saves its own evaluation histories.
            "verb_log": 0,
        }
        if popsize is not None:
            options["popsize"] = int(popsize)
        # Record starts using evaluation numbers beginning at 1, including the first start.
        starts_at.append(before + 1)
        try:
            cma.fmin2(normalized_objective, rng.uniform(size=len(bounds)), params["sigma0"],
                      options=options)
        # The budget may end partway through a population; keep the points already evaluated.
        except BudgetReached:
            pass
        # Avoid an endless restart loop if the optimiser cannot evaluate a point.
        if len(recorder.y) == before:
            raise RuntimeError("CMA-ES stopped without evaluating any points")
    return recorder.result(time.perf_counter() - started,
                           {"restarts": len(starts_at) - 1, "starts_at": starts_at})

## TuRBO-1: `run_turbo1`

TuRBO-1 builds a model near a promising point and searches a region around it. That region expands after progress and shrinks when progress stalls. This function uses Uber Research's implementation and needs NumPy, **PyTorch** (`torch`), **GPyTorch** (`gpytorch`), and **TuRBO** (`turbo`).

Pass the shared `objective`, `bounds`, `budget`, and `seed`. Its `params` dictionary must include:

- **`n_initial_points`** is a positive integer setting the number of initial points spread across the coordinate ranges.
- **`batch_size`** is a positive integer setting how many candidates are proposed together. One gives sequential proposals in this experiment.
- **`use_ard`** is a Boolean, which lets the model learn a different length scale for each coordinate.
- **`n_training_steps`** is an integer of at least 30, setting the number of training steps when fitting the model.
- **`verbose`** is a Boolean controlling whether TuRBO prints its own progress messages.
- **`device`** is `"cpu"` or `"cuda"`
- **`dtype`** is `"float64"` or `"float32"`, selecting numerical precision

The budget must be **strictly larger** than both `n_initial_points` and `batch_size`. 

```python
turbo_params = {
    "n_initial_points": 4,
    "batch_size": 1,
    "use_ard": True,
    "n_training_steps": 50,
    "verbose": False,
    "device": "cpu",
    "dtype": "float64",
}
result = run_turbo1(objective, bounds, budget=50, seed=42, params=turbo_params)
```

The function seeds NumPy and PyTorch before starting. Initialization and TuRBO's own restarts count towards the same budget.
Reference: [Turbo1 implementation and constructor requirements](https://github.com/uber-research/TuRBO/blob/master/turbo/turbo_1.py).


In [ ]:
def run_turbo1(objective, bounds, budget, seed, params):
    """Run Uber's TuRBO-1 with seeded randomness and a hard evaluation cap."""
    import torch
    from turbo import Turbo1

    bounds = _validated_bounds(bounds)
    recorder = EvaluationResults(objective, budget, bounds)
    started = time.perf_counter()
    # Seed both libraries because TuRBO uses NumPy sampling and PyTorch model fitting.
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        optimiser = Turbo1(
            # Record all calls, including TuRBO's initial samples and internal restarts.
            f=recorder,
            lb=bounds[:, 0], ub=bounds[:, 1],
            n_init=params["n_initial_points"],
            max_evals=budget,
            batch_size=params["batch_size"],
            use_ard=params["use_ard"],
            n_training_steps=params["n_training_steps"],
            verbose=params["verbose"],
            device=params["device"], dtype=params["dtype"],
        )
        optimiser.optimize()
    # Stop cleanly if a batch or restart would ask for more than the allowed calls.
    except BudgetReached:
        pass
    return recorder.result(time.perf_counter() - started)


## Uniform random search: `run_uniform_random`

Random search draws each point uniformly and independently from the bounds. It does not use earlier values to choose later points. This is the simplest baseline and needs **NumPy only**.

It takes the same shared inputs, with an empty parameter dictionary:

```python
result = run_uniform_random(objective, bounds, budget=50, seed=42, params={})
```

`params` is accepted to keep consistency, but Random does not read it.


In [ ]:
def run_uniform_random(objective, bounds, budget, seed, params):
    """Evaluate independent uniform points; params is unused and may be {}."""
    bounds = _validated_bounds(bounds)
    recorder = EvaluationResults(objective, budget, bounds)
    started = time.perf_counter()
    # Use a local seeded generator so repeated runs with this seed reproduce the same points.
    rng = np.random.default_rng(seed)
    # Sample every coordinate independently within its bounds, without using earlier outcomes.
    for _ in range(budget):
        recorder(rng.uniform(bounds[:, 0], bounds[:, 1]))
    return recorder.result(time.perf_counter() - started)


## Choosing a method by name: `run_method`

`run_method(name, objective, bounds, budget, seed, params)` chooses one of the four functions above and passes inputs to it. The selections are `"BO"`, `"CMA-ES"`, `"TuRBO-1"`, and `"Random"`. 

In [ ]:
def run_method(name, objective, bounds, budget, seed, params):
    """Call the selected method with shared inputs and return its result dictionary."""
    # All wrappers expose the same inputs and return the same result fields.
    methods = {
        "BO": run_bo,
        "CMA-ES": run_cma_es,
        "TuRBO-1": run_turbo1,
        "Random": run_uniform_random,
    }
    try:
        method = methods[name]
    except KeyError as error:
        raise ValueError(f"Unknown method: {name!r}") from error
    # Give the selected wrapper a copy of its settings to protect the shared configuration.
    return method(objective, bounds, budget, seed, dict(params))